In [4]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import balanced_accuracy_score, roc_auc_score

X = pd.read_parquet("../data/processed/luad_X.parquet")
labels = pd.read_csv("../data/processed/luad_labels.csv")

y = labels["label"]

(X.index == labels["sample_id"]).all(), X.shape, labels["sample_type"].value_counts()

(np.True_,
 (574, 20530),
 sample_type
 tumor     515
 normal     59
 Name: count, dtype: int64)

In [5]:
def select_and_evaluate(
    seed: int,
    X: pd.DataFrame,
    y: pd.Series,
    panel_sizes=(20, 10, 5, 3, 2),
    C=0.1,
):
    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.2,
        random_state=seed,
        stratify=y,
    )

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    selector_model = LogisticRegression(
        penalty="l1",
        solver="saga",
        class_weight="balanced",
        max_iter=10000,
        C=C,
        random_state=seed,
    )

    selector_model.fit(X_train_scaled, y_train)

    coefs = selector_model.coef_[0]

    ranking = pd.DataFrame({
        "gene": X.columns,
        "coef": coefs,
        "abs_coef": np.abs(coefs),
    }).sort_values("abs_coef", ascending=False)

    rows = []
    gene_rows = []

    for k in panel_sizes:
        selected_genes = ranking.head(k)["gene"].tolist()

        # evaluate selected panel
        X_train_panel = X_train[selected_genes]
        X_test_panel = X_test[selected_genes]

        panel_scaler = StandardScaler()
        X_train_panel_scaled = panel_scaler.fit_transform(X_train_panel)
        X_test_panel_scaled = panel_scaler.transform(X_test_panel)

        model = LogisticRegression(
            max_iter=5000,
            class_weight="balanced",
            random_state=seed,
        )

        model.fit(X_train_panel_scaled, y_train)

        y_pred = model.predict(X_test_panel_scaled)
        y_prob = model.predict_proba(X_test_panel_scaled)[:, 1]

        rows.append({
            "seed": seed,
            "panel_size": k,
            "balanced_accuracy": balanced_accuracy_score(y_test, y_pred),
            "roc_auc": roc_auc_score(y_test, y_prob),
            "selected_genes": ",".join(selected_genes),
        })

        for rank, gene in enumerate(selected_genes, start=1):
            gene_rows.append({
                "seed": seed,
                "panel_size": k,
                "rank": rank,
                "gene": gene,
                "coef": ranking.loc[ranking["gene"] == gene, "coef"].iloc[0],
                "abs_coef": ranking.loc[ranking["gene"] == gene, "abs_coef"].iloc[0],
            })

    return pd.DataFrame(rows), pd.DataFrame(gene_rows)

In [6]:
import os

from joblib import Parallel, delayed

n_jobs = min(30, os.cpu_count() or 1)

outputs = Parallel(n_jobs=n_jobs, verbose=10)(
    delayed(select_and_evaluate)(seed, X, y) for seed in range(30)
)

all_results, all_genes = zip(*outputs)
stability_results = pd.concat(all_results, ignore_index=True)
stability_genes = pd.concat(all_genes, ignore_index=True)

[Parallel(n_jobs=16)]: Using backend LokyBackend with 16 concurrent workers.
/Users/adityaanand/dev/cancer-lab/.venv/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
/Users/adityaanand/dev/cancer-lab/.venv/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:1429: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(
/Users/adityaanand/dev/cancer-lab/.venv/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1

In [7]:
performance_summary = (
    stability_results
    .groupby("panel_size")
    .agg(
        mean_bal_acc=("balanced_accuracy", "mean"),
        std_bal_acc=("balanced_accuracy", "std"),
        min_bal_acc=("balanced_accuracy", "min"),
        max_bal_acc=("balanced_accuracy", "max"),
        mean_auc=("roc_auc", "mean"),
        std_auc=("roc_auc", "std"),
        min_auc=("roc_auc", "min"),
        max_auc=("roc_auc", "max"),
    )
    .reset_index()
)

performance_summary

,panel_size,mean_bal_acc,std_bal_acc,min_bal_acc,max_bal_acc,mean_auc,std_auc,min_auc,max_auc
0,2,0.977211,0.025298,0.887540,1.0,0.996980,0.003605,0.987055,1.0
1,3,0.981769,0.019011,0.948625,1.0,0.998355,0.002253,0.991909,1.0
2,5,0.978964,0.025909,0.911812,1.0,0.999056,0.001489,0.993528,1.0
3,10,0.976510,0.026538,0.911812,1.0,0.999353,0.000960,0.996764,1.0
4,20,0.986947,0.021638,0.916667,1.0,0.999595,0.000697,0.997573,1.0


In [8]:
gene_frequency = (
    stability_genes
    .groupby(["panel_size", "gene"])
    .size()
    .reset_index(name="selection_count")
)

n_splits = stability_genes["seed"].nunique()

gene_frequency["selection_frequency"] = (
    gene_frequency["selection_count"] / n_splits
)

gene_frequency = gene_frequency.sort_values(
    ["panel_size", "selection_frequency"],
    ascending=[True, False],
)

gene_frequency.head(50)

,panel_size,gene,selection_count,selection_frequency
5,2,PYCR1,28,0.933333
2,2,CD5L,13,0.433333
1,2,ANGPT4,7,0.233333
3,2,ETV4,6,0.200000
0,2,ADM2,5,0.166667
4,2,OR6K3,1,0.033333
16,3,PYCR1,29,0.966667
8,3,CD5L,21,0.700000
7,3,ANGPT4,14,0.466667
11,3,ETV4,9,0.300000


In [9]:
for k in [2, 3, 5, 10, 20]:
    print(f"\nPanel size {k}")
    display(
        gene_frequency[gene_frequency["panel_size"] == k]
        .sort_values("selection_frequency", ascending=False)
        .head(20)
    )


Panel size 2


,panel_size,gene,selection_count,selection_frequency
5,2,PYCR1,28,0.933333
2,2,CD5L,13,0.433333
1,2,ANGPT4,7,0.233333
3,2,ETV4,6,0.200000
0,2,ADM2,5,0.166667
4,2,OR6K3,1,0.033333



Panel size 3


,panel_size,gene,selection_count,selection_frequency
16,3,PYCR1,29,0.966667
8,3,CD5L,21,0.700000
7,3,ANGPT4,14,0.466667
11,3,ETV4,9,0.300000
6,3,ADM2,8,0.266667
14,3,OR6K3,2,0.066667
15,3,PLEKHN1,2,0.066667
9,3,CELA2B,1,0.033333
10,3,EMP2,1,0.033333
12,3,HBM,1,0.033333



Panel size 5


,panel_size,gene,selection_count,selection_frequency
29,5,PYCR1,30,1.000000
19,5,ANGPT4,25,0.833333
21,5,CD5L,25,0.833333
24,5,ETV4,18,0.600000
18,5,ADM2,8,0.266667
20,5,ATP10B,8,0.266667
27,5,OR6K3,8,0.266667
26,5,LGR4,6,0.200000
28,5,PLEKHN1,6,0.200000
22,5,CELA2B,5,0.166667



Panel size 10


,panel_size,gene,selection_count,selection_frequency
58,10,PYCR1,30,1.000000
39,10,CD5L,29,0.966667
33,10,ANGPT4,29,0.966667
44,10,ETV4,27,0.900000
34,10,ATP10B,22,0.733333
32,10,ADM2,19,0.633333
49,10,LGR4,19,0.633333
53,10,OR6K3,19,0.633333
57,10,PLEKHN1,16,0.533333
42,10,EMP2,14,0.466667



Panel size 20


,panel_size,gene,selection_count,selection_frequency
66,20,ANGPT4,30,1.000000
110,20,PYCR1,30,1.000000
75,20,CD5L,30,1.000000
85,20,ETV4,30,1.000000
67,20,ATP10B,29,0.966667
102,20,OR6K3,29,0.966667
63,20,ADM2,27,0.900000
82,20,EMP2,27,0.900000
109,20,PLEKHN1,27,0.900000
113,20,RTKN2,26,0.866667


In [10]:
stability_results.to_csv(
    "../data/processed/repeated_split_panel_performance.csv",
    index=False,
)

stability_genes.to_csv(
    "../data/processed/repeated_split_selected_genes.csv",
    index=False,
)

gene_frequency.to_csv(
    "../data/processed/repeated_split_gene_frequency.csv",
    index=False,
)

performance_summary.to_csv(
    "../data/processed/repeated_split_performance_summary.csv",
    index=False,
)